# ModelOff 2017 Finals - Funding Fun

## Financial Model: 30-Year Infrastructure Concession

This notebook implements a complete project finance model for a 30-year infrastructure
concession. It reads scenario data from the provided Excel workbook and solves for optimal
financing parameters across 10 input scenarios and 10 operating cash flow scenarios.

In [ ]:
import pyxlsb
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Install pyxlsb if needed
try:
    import pyxlsb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyxlsb', '-q'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    import pyxlsb

print('Imports complete')

In [ ]:
###############################################################################
# DATA LOADING - Read scenario parameters and CFADS from .xlsb workbook
###############################################################################

def xl2date(serial):
    """Convert Excel serial number to datetime."""
    return datetime(1899, 12, 30) + timedelta(days=int(serial))

xlsb_path = '/workspace/data/MO2017-Finals-Sec-2-Funding-Fun.xlsb'
with pyxlsb.open_workbook(xlsb_path) as wb:
    with wb.get_sheet('Inputs') as sh:
        INP = [list(item.v for item in row) for row in sh.rows()]
    with wb.get_sheet('Operations') as sh:
        OPS = [list(item.v for item in row) for row in sh.rows()]

# Quarter end dates from Operations sheet
qtr_serials = [v for v in OPS[5][9:] if v is not None and isinstance(v, (int, float))]
QD = [xl2date(v) for v in qtr_serials]
NQ = len(QD)

# Asset life flag (1 = asset is active in this quarter)
AF = [float(v) if v is not None else 0.0 for v in OPS[11][9:9+NQ]]

# CFADS by Operating Scenario (1-indexed), zeroed outside asset life
CFADS = {}
for os_i in range(10):
    row = OPS[20 + os_i]
    c = [float(row[j]) if j < len(row) and row[j] is not None else 0.0 for j in range(9, 9+NQ)]
    CFADS[os_i+1] = [x*f for x, f in zip(c, AF)]

# Key dates and indices
PURCHASE_DATE = xl2date(43100)   # 2017-12-31
ASSET_EXPIRY = xl2date(54057)    # 2047-12-31
DEPOSIT_RATE = 0.02
OVERDRAFT_RATE = 0.07
REFI_DATES = [xl2date(int(OPS[4][9+j] if OPS[4][9+j] else 0)) 
              for j in range(NQ)]  # placeholder
# Actually read from Inputs sheet
REFI_DATES = [xl2date(int(INP[35][9])), xl2date(int(INP[36][9])), xl2date(int(INP[37][9]))]

PI = next(i for i, d in enumerate(QD) if d >= PURCHASE_DATE)  # Purchase quarter index
EI = next(i for i, d in enumerate(QD) if d >= ASSET_EXPIRY)   # Expiry quarter index
MI = EI - int(INP[30][9])  # Maturity = expiry - debt tail quarters
EARLIEST_REPAY = PI + 1  # At least 1 quarter (3 months) after purchase

# Parse 10 input scenarios
def ginp(r, c):
    if r < len(INP) and c < len(INP[r]): return INP[r][c]
    return None

SCENARIOS = {}
for s in range(1, 11):
    col = 12 + s  # S1=col13, S2=col14, ...
    fr_raw = ginp(52, col); dc_raw = ginp(53, col)
    db_raw = ginp(62, col); eq_raw = ginp(63, col); rr_raw = ginp(65, col)
    SCENARIOS[s] = {
        'rate': float(ginp(33, col)), 'rfee': float(ginp(39, col)),
        'os': int(ginp(23, col)),
        'fr': xl2date(fr_raw) if isinstance(fr_raw, (int, float)) else None,
        'dscr': float(dc_raw) if isinstance(dc_raw, (int, float)) else None,
        'debt': float(db_raw) if isinstance(db_raw, (int, float)) else None,
        'eq': float(eq_raw) if isinstance(eq_raw, (int, float)) else None,
        'rrr': float(rr_raw) if isinstance(rr_raw, (int, float)) else None,
    }

print(f'Loaded {NQ} quarters, {len(CFADS)} OS, {len(SCENARIOS)} scenarios')
print(f'Purchase: {PURCHASE_DATE.date()}, Expiry: {ASSET_EXPIRY.date()}, Maturity: {QD[MI].date()}')

In [ ]:
###############################################################################
# FINANCIAL MODEL ENGINE
###############################################################################

def days_in_qtr(dt):
    """Number of days in the quarter ending on dt."""
    q = (dt.month - 1) // 3
    sm = q * 3 + 1
    return (dt - datetime(dt.year, sm, 1)).days + 1

def xirr(cfs, dates, guess=0.1):
    """Calculate XIRR using bisection."""
    if len(cfs) < 2:
        return 0.0
    d0 = dates[0]
    def npv(r):
        s = 0
        for cf, d in zip(cfs, dates):
            t = (d - d0).days / 365.0
            try:
                s += cf / (1 + r) ** t
            except (OverflowError, ZeroDivisionError):
                return 1e10 if r < 0 else -1e10
        return s
    lo, hi = -0.5, 2.0
    if npv(lo) * npv(hi) > 0:
        return guess
    for _ in range(200):
        m = (lo + hi) / 2
        if npv(m) * npv(lo) < 0:
            hi = m
        else:
            lo = m
        if hi - lo < 1e-13:
            break
    return (lo + hi) / 2

def d2i(dt):
    """Convert date to quarter index."""
    for i, d in enumerate(QD):
        if d >= dt:
            return i
    return NQ - 1

def run_model(debt, equity, dscr, fri, rate, rfee, cfads):
    """
    Run quarterly cash flow model.
    
    Mechanics:
    - Purchase quarter: debt drawn, equity invested, cash = 0
    - Each quarter: interest accrues, sculpted principal repayment
    - Refinancing fee capitalized into debt (not paid from cash)
    - Distributions on June 30 and at asset expiry
    - Iterates for convergence (cash balance affects interest)
    """
    if equity < 0.01:
        return {'irr': 999, 'repaid': False, 'pp': debt + equity,
                'debt': debt, 'eq': equity, 'dscr': dscr}
    
    db = [0.0] * NQ
    cb = [0.0] * NQ
    pr = [0.0] * NQ
    di = [0.0] * NQ
    
    for iteration in range(50):
        for i in range(NQ):
            dt = QD[i]
            dq = days_in_qtr(dt)
            
            if i < PI:
                db[i] = cb[i] = pr[i] = di[i] = 0
                continue
            if i == PI:
                db[i] = debt
                cb[i] = pr[i] = di[i] = 0
                continue
            
            od = db[i-1]  # opening debt
            oc = cb[i-1]  # opening cash
            
            # Interest
            dint = od * rate * dq / 365
            cint = oc * (DEPOSIT_RATE if oc >= 0 else OVERDRAFT_RATE) * dq / 365
            nie = dint - cint  # net interest expense
            
            # Sculpted principal repayment
            p = 0
            if fri <= i <= MI and od > 0.001:
                p = cfads[i] / dscr - nie
                p = max(0, min(p, od))
            pr[i] = p
            
            # Refinancing fee (capitalized into debt, NOT paid from cash)
            dap = od - p
            rc = 0
            for rd in REFI_DATES:
                if dt == rd and dap > 0.001:
                    rc = dap * rfee
            db[i] = dap + rc
            
            # Cash waterfall (refi fee is capitalized, not deducted from cash)
            cf = oc + cfads[i] + cint - dint - p
            
            # Distribution on June 30 or at asset expiry
            d = 0
            if ((dt.month == 6 and dt.day == 30) or dt == ASSET_EXPIRY) and cf > 0:
                d = cf
                cf = 0
            di[i] = d
            cb[i] = cf
    
    # Last repayment date
    lrd = None
    for i in range(NQ - 1, -1, -1):
        if pr[i] > 0.001:
            lrd = QD[i]
            break
    
    repaid = db[MI] < 0.5
    
    # Equity IRR
    ecf = [-equity]
    edt = [PURCHASE_DATE]
    for i in range(NQ):
        if di[i] > 0.001:
            ecf.append(di[i])
            edt.append(QD[i])
    irr = xirr(ecf, edt)
    
    return {
        'irr': irr, 'lrd': lrd, 'repaid': repaid,
        'dmat': db[MI], 'pp': debt + equity,
        'debt': debt, 'eq': equity, 'dscr': dscr,
        'td': sum(di)
    }

print('Model engine defined')

In [ ]:
###############################################################################
# SOLVER - Finds optimal values for unknown scenario parameters
###############################################################################

def _max_eq(debt, dscr, fri, rate, rfee, cf, rrr, iters=60):
    """Binary search for max equity where IRR >= RRR and debt repaid."""
    lo, hi = 0.0, 200000.0
    best = None
    for _ in range(iters):
        mid = (lo + hi) / 2
        r = run_model(debt, mid, dscr, fri, rate, rfee, cf)
        if r['repaid'] and r['irr'] >= rrr - 1e-9:
            lo = mid; best = r
        else:
            hi = mid
    return best

def _max_debt(equity, dscr, fri, rate, rfee, cf, rrr, iters=60):
    """Binary search for max debt where IRR >= RRR and debt repaid."""
    lo, hi = 0.0, 200000.0
    best = None
    for _ in range(iters):
        mid = (lo + hi) / 2
        r = run_model(mid, equity, dscr, fri, rate, rfee, cf)
        if r['repaid'] and r['irr'] >= rrr - 1e-9:
            lo = mid; best = r
        else:
            hi = mid
    return best

def _pp_at_debt(dv, dscr, fri, rate, rfee, cf, rrr):
    """Compute max PP at a given debt level."""
    lb = _max_eq(dv, dscr, fri, rate, rfee, cf, rrr, 50)
    return lb['pp'] if lb else 0

def _max_pp_debt_eq(dscr, fri, rate, rfee, cf, rrr):
    """Find max PP using ternary search on debt (PP is unimodal in debt)."""
    # Ternary search on debt to find the peak of PP = debt + max_equity(debt)
    lo, hi = 0.0, 200000.0
    for _ in range(80):
        if hi - lo < 0.1:
            break
        m1 = lo + (hi - lo) / 3
        m2 = hi - (hi - lo) / 3
        p1 = _pp_at_debt(m1, dscr, fri, rate, rfee, cf, rrr)
        p2 = _pp_at_debt(m2, dscr, fri, rate, rfee, cf, rrr)
        if p1 < p2:
            lo = m1
        else:
            hi = m2
    # Fine binary search around the peak
    best_debt = (lo + hi) / 2
    best = _max_eq(best_debt, dscr, fri, rate, rfee, cf, rrr, 60)
    # Also check nearby to handle any non-smoothness
    for offset in [-50, -10, -1, 1, 10, 50]:
        dv = best_debt + offset
        if dv < 0:
            continue
        lb = _max_eq(dv, dscr, fri, rate, rfee, cf, rrr, 60)
        if lb and (best is None or lb['pp'] > best['pp']):
            best = lb
    return best

def solve(sc_num, os_override=None):
    """Solve a scenario, optionally overriding the operating scenario."""
    sc = SCENARIOS[sc_num]
    os_n = os_override if os_override else sc['os']
    cf = CFADS[os_n]
    rate, rfee = sc['rate'], sc['rfee']
    fr, dscr, debt, eq, rrr = sc['fr'], sc['dscr'], sc['debt'], sc['eq'], sc['rrr']
    fri = d2i(fr) if fr else None
    need = [k for k in ['fr', 'dscr', 'debt', 'eq', 'rrr'] if sc[k] is None]

    # Only RRR unknown: just compute IRR
    if need == ['rrr']:
        return run_model(debt, eq, dscr, fri, rate, rfee, cf)

    # First repay + RRR: find latest repay date where debt is repaid
    if set(need) == {'fr', 'rrr'}:
        for idx in range(MI, EARLIEST_REPAY - 1, -1):
            r = run_model(debt, eq, dscr, idx, rate, rfee, cf)
            if r['repaid']:
                r['fr_date'] = QD[idx]
                return r
        return None

    # Equity only: binary search max equity where IRR >= RRR
    if need == ['eq']:
        return _max_eq(debt, dscr, fri, rate, rfee, cf, rrr, 80)

    # DSCR + RRR: binary search max DSCR where debt repaid
    if set(need) == {'dscr', 'rrr'}:
        lo, hi = 1.0, 5.0
        best = None
        for _ in range(80):
            mid = (lo + hi) / 2
            r = run_model(debt, eq, mid, fri, rate, rfee, cf)
            if r['repaid']:
                lo = mid; best = r
            else:
                hi = mid
        return best

    # First repay + equity: maximize PP over all repay dates
    if set(need) == {'fr', 'eq'}:
        best = None
        for idx in range(EARLIEST_REPAY, MI + 1):
            lb = _max_eq(debt, dscr, idx, rate, rfee, cf, rrr, 60)
            if lb:
                lb['fr_date'] = QD[idx]
                if best is None or lb['pp'] > best['pp']:
                    best = lb
        return best

    # DSCR + equity: coarse+fine search on DSCR, binary search equity
    if set(need) == {'dscr', 'eq'}:
        best = None
        for d10k in range(10000, 50000, 100):
            d = d10k / 10000.0
            lb = _max_eq(debt, d, fri, rate, rfee, cf, rrr, 50)
            if lb and (best is None or lb['pp'] > best['pp']):
                best = lb
        if best:
            cd = best['dscr']
            for d10k in range(int((cd - 0.02) * 10000), int((cd + 0.02) * 10000) + 1):
                d = d10k / 10000.0
                lb = _max_eq(debt, d, fri, rate, rfee, cf, rrr, 60)
                if lb and lb['pp'] > best['pp']:
                    best = lb
        return best

    # DSCR + debt: coarse+fine search on DSCR, binary search debt
    if set(need) == {'dscr', 'debt'}:
        best = None
        for d10k in range(10000, 50000, 100):
            d = d10k / 10000.0
            lb = _max_debt(eq, d, fri, rate, rfee, cf, rrr, 50)
            if lb and (best is None or lb['pp'] > best['pp']):
                best = lb
        if best:
            cd = best['dscr']
            for d10k in range(int((cd - 0.02) * 10000), int((cd + 0.02) * 10000) + 1):
                d = d10k / 10000.0
                lb = _max_debt(eq, d, fri, rate, rfee, cf, rrr, 60)
                if lb and lb['pp'] > best['pp']:
                    best = lb
        return best

    # Debt + equity: ternary search on debt to maximize PP
    if set(need) == {'debt', 'eq'}:
        return _max_pp_debt_eq(dscr, fri, rate, rfee, cf, rrr)

    return None

print('Solver defined')

In [ ]:
###############################################################################
# SOLVE Q1-Q20 (Scenarios 1-10 with their default Operating Scenarios)
###############################################################################

results = {}
for s in range(1, 11):
    r = solve(s)
    results[s] = r
    if r:
        need = [k for k in ['fr','dscr','debt','eq','rrr'] if SCENARIOS[s][k] is None]
        fr_str = r.get('fr_date', r.get('lrd'))
        fr_str = fr_str.strftime('%d %b %Y') if fr_str else 'N/A'
        print(f"S{s}: IRR={r['irr']*100:.3f}% DSCR={r['dscr']:.3f} "
              f"Debt={r['debt']:.0f} Eq={r['eq']:.0f} PP={r['pp']:.0f} "
              f"Date={fr_str} solve={need}")

In [ ]:
###############################################################################
# MAP RESULTS TO Q1-Q20 ANSWERS
###############################################################################

# Q1-Q3: Scenario 1 (multiple choice)
q1_options = {'A':'31 Dec 2044','B':'31 Mar 2045','C':'30 Jun 2045','D':'30 Sep 2045',
              'E':'31 Dec 2045','F':'31 Mar 2046','G':'30 Jun 2046','H':'30 Sep 2046','I':'31 Dec 2046'}
r1 = results[1]
lrd_str = r1['lrd'].strftime('%d %b %Y')
q1 = next((k for k, v in q1_options.items() if v == lrd_str), lrd_str)

q2_options = {'A':66929,'B':66939,'C':66949,'D':66959,'E':66969,'F':66979,'G':66989,'H':66999,'I':67009}
td = round(r1['td'])
q2 = next((k for k, v in q2_options.items() if v == td), str(td))

q3_options = {'A':'9.386%','B':'9.387%','C':'9.388%','D':'9.389%','E':'9.390%',
              'F':'9.391%','G':'9.392%','H':'9.393%','I':'9.394%'}
irr_str = f"{r1['irr']*100:.3f}%"
q3 = next((k for k, v in q3_options.items() if v == irr_str), irr_str)

print(f"Q1: {q1} ({lrd_str})")
print(f"Q2: {q2} ({td})")
print(f"Q3: {q3} ({irr_str})")

# Q4-Q20: Free field answers
def fmt_date(r, key='fr_date'):
    d = r.get(key, r.get('lrd'))
    return d.strftime('%d %b %Y') if d else 'N/A'

def fmt_irr(r):
    return f"{r['irr']*100:.3f}%"

def fmt_k(val):
    """Format value in $'000 - round to nearest integer."""
    return f"{round(val)} k"

q4 = fmt_date(results[2])
q5 = fmt_irr(results[2])
q6 = fmt_k(results[3]['eq'])
q7 = f"{results[4]['dscr']:.3f}"
q8 = fmt_irr(results[4])
q9 = fmt_date(results[5])
q10 = fmt_k(results[5]['eq'])
q11 = f"{results[6]['dscr']:.3f}"
q12 = fmt_k(results[6]['eq'])
q13 = fmt_date(results[7])
q14 = fmt_irr(results[7])
q15 = f"{results[8]['dscr']:.3f}"
q16 = fmt_irr(results[8])
q17 = f"{results[9]['dscr']:.3f}"
q18 = fmt_k(results[9]['debt'])

# S10 (debt+equity solver): debt uses int() since it's at upper feasibility boundary
# equity derived as round(pp) - int(debt)
q19 = f"{int(results[10]['debt'])} k"
q20 = f"{round(results[10]['pp']) - int(results[10]['debt'])} k"

for i, v in enumerate([q4,q5,q6,q7,q8,q9,q10,q11,q12,q13,q14,q15,q16,q17,q18,q19,q20], 4):
    print(f"Q{i}: {v}")

In [ ]:
###############################################################################
# Q21-Q23: Cross-scenario analysis (vary Operating Scenario)
###############################################################################

# Q21: S5 with each OS 1-10, find largest Equity Investment
print("Q21: S5 with each OS, find largest equity...")
q21_results = {}
for os_n in range(1, 11):
    r = solve(5, os_override=os_n)
    if r:
        q21_results[os_n] = round(r['eq'])
        print(f"  OS{os_n}: Eq={round(r['eq'])}")
best_os21 = max(q21_results, key=q21_results.get)
q21 = f"OS {best_os21}, {q21_results[best_os21]} k"
print(f"  => {q21}")

# Q22: S9 with each OS 1-10, find 3rd largest DSCR
print("\nQ22: S9 with each OS, 3rd largest DSCR...")
q22_results = {}
for os_n in range(1, 11):
    r = solve(9, os_override=os_n)
    if r:
        q22_results[os_n] = round(r['dscr'], 3)
        print(f"  OS{os_n}: DSCR={r['dscr']:.3f}")
ranked22 = sorted(q22_results.items(), key=lambda x: x[1], reverse=True)
third_os, third_dscr = ranked22[2]
q22 = f"OS {third_os}, {third_dscr:.3f}"
print(f"  Ranked: {ranked22}")
print(f"  => 3rd: {q22}")

# Q23: S10 with each OS 1-10, find 6th largest Purchase Price
print("\nQ23: S10 with each OS, 6th largest PP...")
q23_results = {}
for os_n in range(1, 11):
    r = solve(10, os_override=os_n)
    if r:
        pp_rounded = round(r['pp'])
        q23_results[os_n] = pp_rounded
        d_int = int(r['debt'])
        e_derived = pp_rounded - d_int
        print(f"  OS{os_n}: PP={pp_rounded}, D={d_int}, E={e_derived}")
ranked23 = sorted(q23_results.items(), key=lambda x: x[1], reverse=True)
sixth_os, sixth_pp = ranked23[5]
q23 = f"OS {sixth_os}, {sixth_pp} k"
print(f"  Ranked: {ranked23}")
print(f"  => 6th: {q23}")

In [ ]:
###############################################################################
# FINAL ANSWERS
###############################################################################

answers = {
    'question1': q1,
    'question2': q2,
    'question3': q3,
    'question4': q4,
    'question5': q5,
    'question6': q6,
    'question7': q7,
    'question8': q8,
    'question9': q9,
    'question10': q10,
    'question11': q11,
    'question12': q12,
    'question13': q13,
    'question14': q14,
    'question15': q15,
    'question16': q16,
    'question17': q17,
    'question18': q18,
    'question19': q19,
    'question20': q20,
    'question21': q21,
    'question22': q22,
    'question23': q23,
}

print('\nFinal Answers:')
for k, v in answers.items():
    print(f'  {k}: {v}')